# Corpus FR — générer les dialogues de tool calling (texte, Gemini) → Hub

**Runtime : CPU suffit.** Secrets : `HF_TOKEN` (écriture), **`GEMINI_API_KEY`**.

La brique `fr_toolcalling` du plan 150 h vise **2 500 dialogues** ; `tc_fr_v1.jsonl` en tient
une première tranche. Cette cellule génère la suite avec le générateur du repo
(`lfm2-generate-data --lang fr --mode loop`) : énoncés et réponses parlées en français,
outils et valeurs d'arguments inchangés (un appel est structurel, pas linguistique),
chaque cas vérifié par le parseur et le registre avant écriture.

Le fichier part sur `Rcarvalo/lfm25-fr-corpus-v1` dès que le générateur rend la main.
**Ensuite** : dans `colab_corpus_assistant_waves.ipynb`, ajouter
`corpus/TC_fr/tc_fr_v2.jsonl` à `BRICK_A_SOURCES` pour le faire parler par la voix
`fr_female`.

Coût : quelques centimes de Gemini Flash. Si des 429 apparaissent, baissez `--concurrency`.


In [ ]:
# Jetons — le plus propre : Colab > icône clé > secrets HF_TOKEN (écriture), GEMINI_API_KEY, WANDB_API_KEY (optionnel).
import os
from getpass import getpass
try:
    from google.colab import userdata
    read = userdata.get
except Exception:
    read = lambda name: getpass(f"{name} : ")
for name, required in (("HF_TOKEN", True), ("GEMINI_API_KEY", False), ("WANDB_API_KEY", False)):
    try:
        value = read(name)
    except Exception:
        value = "" if not required else getpass(f"{name} : ")
    if value:
        os.environ[name] = value
    elif required:
        raise SystemExit(f"{name} manquant")
print("jetons chargés :", [n for n in ("HF_TOKEN", "GEMINI_API_KEY", "WANDB_API_KEY") if os.environ.get(n)])


## Générer et pousser (≈ 15-30 min selon le quota Gemini)

In [ ]:
# Reprenable : un fichier déjà complet est simplement re-poussé.
import os, subprocess, urllib.request
os.environ.update({
    "LFM2_BRANCH": "rd/pr_rca_eval_baseline",
    "LFM2_JOB": "generate_tc_fr",
    "LFM2_ARGS": "--n-total 2500 --output corpus/TC_fr/tc_fr_v2.jsonl --concurrency 6",
    "LFM2_EXTRAS": "serving-liquid,eval,inspect",
    "LFM2_ROOT": "/content/repo",
    "LFM2_OUT": "/content/out"
})
urllib.request.urlretrieve("https://raw.githubusercontent.com/rcarvalo/finetuning_s2s_toolcalling/rd/pr_rca_eval_baseline/infra/colab_entrypoint.sh", "/content/entry.sh")
# Au PREMIER PLAN, volontairement : le kernel occupé est ce qui garde la session Colab vivante.
subprocess.run(["bash", "/content/entry.sh"], check=False)
